# Evidence-bound MCP release orchestration

A green CI job is not a deployment proof. In this lab you will audit a workflow, correlate authenticated evidence, approve one exact plan, deploy with short-lived identity, reconcile an ambiguous result, and verify success or rollback from observable state. Everything is deterministic and local; no cloud or registry is changed.

## Learning objectives

By the end you can explain the three workflow trust zones; bind source, build, test, supply-chain, approval, identity, deployment, health, and recovery evidence; and distinguish transient failures from unknown outcomes that require reconciliation.

In [ ]:
import json
import runpy
from dataclasses import replace
from pathlib import Path

lab = runpy.run_path(Path('lab.py'))
env = lab['build_demo_environment']()
RunState = lab['RunState']
DeployResult = lab['DeployResult']


## 1. Audit workflow trust zones

Pull-request tests are unprivileged, the protected-main build can publish and attest, and production deploy uses a protected environment plus OIDC. Each third-party action is pinned to an allowlisted full commit SHA.

In [ ]:
contract_digest = env.controller.auditor.verify_contract(env.contract)
[{
    'phase': job.phase,
    'trigger': job.trigger,
    'permissions': job.permissions,
    'environment': job.environment,
    'oidc': job.requests_oidc,
    'checkout': job.checkout_ref,
} for job in env.contract.jobs], contract_digest


A familiar action tag is still mutable. The static gate rejects it before release state exists. The same gate rejects direct workflow expressions inside shell source, over-broad token permissions, privileged cache writes, and non-ephemeral release runners.

In [ ]:
data = env.contract.model_dump(mode='json')
data['jobs'][0]['steps'][0]['uses'] = 'actions/checkout@v6'
unsafe_contract = lab['WorkflowContract'].model_validate(data)
try:
    env.controller.auditor.verify_contract(unsafe_contract)
except lab['PipelineDenied'] as denied:
    print(denied.code, denied)


## 2. Bind runtime identity

The controller checks immutable repository and owner IDs, source repository, protected ref, workflow path and SHA, environment, and manual actor. Names alone are not identity.

In [ ]:
env.context.model_dump(mode='json')


## 3. Advance only with authenticated, correlated evidence

Distinct Ed25519 identities sign protected-source, build, test, supply-chain, rollback, and health evidence. The trust store validates role and signature; the controller then validates subject, freshness, and stage order.

In [ ]:
record = env.controller.start(env.source, env.now)
record = env.controller.record_build(record.version, lab['ARTIFACT_BYTES'], env.build, env.now)
record = env.controller.record_tests(record.version, env.tests, env.now)
{'state': record.state, 'version': record.version, 'digest': record.artifact_digest}


In [ ]:
record = env.controller.verify_release(
    record.version, env.source, env.build, env.tests, env.security, env.rollback, env.now
)
record.plan.model_dump(mode='json')


## 4. Approve the exact plan once

The trusted approval store binds the canonical plan digest, run, approver role, and expiry. Mutation changes the digest; replay fails because consumption is atomic.

In [ ]:
record, receipt = env.controller.approve(record.version, 'approver@example.com', env.now)
{
    'state': record.state,
    'plan_digest': lab['document_digest'](record.plan),
    'receipt': receipt.model_dump(mode='json'),
}


## 5. Exchange workload identity for target-scoped authority

The broker validates issuer, audience, subject, immutable repository and owner IDs, workflow, protected ref, environment, run, and validity window. `id-token: write` alone is not cloud authorization.

In [ ]:
claims = lab['oidc_claims'](env)
credential = env.controller.credentials.exchange(claims, env.context, record.plan, env.now)
credential.model_dump(mode='json')


## 6. Retry only known-safe failures

A transient result enters a bounded retry state. Every attempt reuses one stable operation ID, so the target can recognize duplicate delivery.

In [ ]:
retry_env = lab['build_demo_environment']()
retry_record = lab['advance_to_verified'](retry_env)
retry_record, retry_receipt = retry_env.controller.approve(
    retry_record.version, 'approver@example.com', retry_env.now
)
retry_record = retry_env.controller.deploy(
    retry_record.version, retry_receipt, lab['oidc_claims'](retry_env),
    DeployResult.TRANSIENT, retry_env.now
)
operation_id = retry_record.operation_id
retry_record = retry_env.controller.retry_deploy(retry_record.version, DeployResult.SUCCESS)
{'state': retry_record.state, 'attempts': retry_record.deploy_attempts,
 'stable_operation_id': retry_record.operation_id == operation_id}


## 7. Reconcile an unknown outcome before retry

A lost response may hide a committed write. The lab simulates that ambiguity, queries status by operation ID, and advances to health verification without applying the effect twice.

In [ ]:
unknown_env = lab['build_demo_environment']()
unknown_record = lab['advance_to_verified'](unknown_env)
unknown_record, unknown_receipt = unknown_env.controller.approve(
    unknown_record.version, 'approver@example.com', unknown_env.now
)
unknown_record = unknown_env.controller.deploy(
    unknown_record.version, unknown_receipt, lab['oidc_claims'](unknown_env),
    DeployResult.UNKNOWN, unknown_env.now
)
ambiguous_state = unknown_record.state
unknown_record = unknown_env.controller.reconcile(unknown_record.version)
unknown_record = unknown_env.controller.verify_health(
    unknown_record.version, lab['health_evidence'](unknown_env, unknown_record),
    unknown_env.now + lab['timedelta'](minutes=2)
)
{'ambiguous': ambiguous_state, 'reconciled': unknown_record.state}


## 8. Prove runtime health or verify rollback

Deployment success is provisional. A separate signer reports the operation, expected and observed digests, named checks, and time. Failed checks restore the exact known-good digest admitted in the release plan.

In [ ]:
rollback_env = lab['build_demo_environment']()
rollback_record = lab['advance_to_verified'](rollback_env)
rollback_record, rollback_receipt = rollback_env.controller.approve(
    rollback_record.version, 'approver@example.com', rollback_env.now
)
rollback_record = rollback_env.controller.deploy(
    rollback_record.version, rollback_receipt, lab['oidc_claims'](rollback_env),
    DeployResult.SUCCESS, rollback_env.now
)
failed_health = lab['health_evidence'](
    rollback_env, rollback_record,
    checks={'service-ready': True, 'tool-smoke': True, 'authz-smoke': False},
)
rollback_record = rollback_env.controller.verify_health(
    rollback_record.version, failed_health,
    rollback_env.now + lab['timedelta'](minutes=2)
)
{'state': rollback_record.state, 'restored_digest': rollback_env.target.current_digest}


## 9. Audit decisions and measure completed outcomes

Audit events use stable reason codes and correlation fields without tokens, signatures, or artifact content. Metrics need labelled denominators: a denial count is not a safe-block rate, and a rollback request is not a verified restoration.

In [ ]:
[event.model_dump(mode='json') for event in rollback_env.controller.audit]


In [ ]:
labelled_unsafe_attempts = 20
correctly_blocked = 19
completed_forbidden_deployments = 1
rollback_attempts = 4
verified_restorations = 4
{
    'safe_block_rate': correctly_blocked / labelled_unsafe_attempts,
    'unsafe_deployment_rate': completed_forbidden_deployments / labelled_unsafe_attempts,
    'rollback_success_rate': verified_restorations / rollback_attempts,
}


## Production mapping

Replace the local contract with parsed GitHub workflow and protection settings; local keys with Sigstore, GitHub artifact attestations, or managed signing; in-memory state with transactional storage; constructed OIDC claims with provider verification; and the simulated target/observer with idempotent deployment and authenticated health APIs. Preserve every subject, freshness, state, and outcome invariant.

## Reflection and exercises

1. Why must an unknown result be reconciled before another deployment call?
2. Which fields must approval bind to prevent cross-environment or stale-policy reuse?
3. What does `id-token: write` prove, and what remains for cloud policy to decide?
4. Add a staging target without letting staging identity, evidence, or approval authorize production.
5. Persist the state machine and inject a process crash at every transition. Which operations need a transactional outbox?
6. Design a canary observation that verifies MCP protocol behavior and authorization denials without producing a consequential tool effect.